# 🔒 GIADA IE-P7 — conferma congelata e costo operativo
Nessun training. Sei operatori locali IE-P6 congelati, formule/LUT e sei ibridi Task35. Prima calibrazione nativa separata, poi 32 nuovi episodi e nuovi test atomici/path.
IE-P6 misurava già il core di membrana completo. Qui si aggiunge una pipeline dichiarata con eventi/RNG indipendenti, trasferimenti, rollout causale e output. Nessuna promozione automatica, nessun claim sulla morfologia intera.
GPU T4 e Internet; nessun dataset esterno. NEURON e CUDA in processi separati.

In [ ]:
import importlib.metadata, json, os, shutil, subprocess, sys
from pathlib import Path
WORK=Path('/kaggle/working/giada_ie_p7');WORK.mkdir(parents=True,exist_ok=True)
REPO,TEACHER=WORK/'giada',WORK/'neuron_as_deep_net'
BRANCH='codex/surrogate-validity-audit'
if not REPO.exists():subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/Zagred47/giada.git',str(REPO)],check=True)
REF=os.environ.get('GIADA_IE_P7_REVISION',BRANCH)
subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin',REF],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
if not TEACHER.exists():subprocess.run(['git','clone','--depth','1','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
TEACHER_REV='074c4666300a8ad246601dab179a97a6942f0f29'
subprocess.run(['git','-C',str(TEACHER),'fetch','--depth','1','origin',TEACHER_REV],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','FETCH_HEAD'],check=True)
try: installed=importlib.metadata.version('NEURON').split('+')[0]
except importlib.metadata.PackageNotFoundError: installed=None
if installed!='8.2.7':subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7'],check=True)
assert importlib.metadata.version('NEURON').split('+')[0]=='8.2.7'
assert shutil.which('nrnivmodl') or (Path(sys.executable).parent/'nrnivmodl').is_file()
print({'revision':REVISION,'teacher':TEACHER_REV,'training':False,'inputs_required':0})


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts/giada_ie_p7_native_operational')
assert not OUTPUT.exists(),f'Output già presente: {OUTPUT}'
LOG=WORK/'process.log'
command=[sys.executable,'-u',str(REPO/'scripts/run_ie_p7_confirmation.py'),'--output',str(OUTPUT),'--teacher',str(TEACHER)]
with subprocess.Popen(command,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as worker,LOG.open('w') as log:
    for line in worker.stdout:
        log.write(line);log.flush()
        if line.startswith('[GIADA IE-P7]'):print(line.rstrip(),flush=True)
    exit_code=worker.wait()
OUTPUT.mkdir(parents=True,exist_ok=True);shutil.copy2(LOG,OUTPUT/'process.log')
path=next((OUTPUT/n for n in ('failure_report.json','final_report.json') if (OUTPUT/n).is_file()),None)
REPORT=json.loads(path.read_text()) if path else {'valid':False,'error':'worker terminated without report','exit':exit_code}
if path is None:(OUTPUT/'failure_report.json').write_text(json.dumps(REPORT,indent=2))
print({k:REPORT.get(k) for k in ('valid','diagnosis','error','model_judged','training_performed')})
print({'family_decisions':REPORT.get('family_decisions'),'core_cells':len(REPORT.get('core_rows',[])),'operational_cells':len(REPORT.get('operational',{}).get('rows',[]))})
if exit_code:print('\n'.join(LOG.read_text(errors='replace').splitlines()[-18:]))
ARCHIVE=shutil.make_archive('/kaggle/working/giada_ie_p7_native_operational','zip',root_dir=OUTPUT)


In [ ]:
DOWNLOAD_IN_BROWSER=False
if DOWNLOAD_IN_BROWSER:
    import base64
    from IPython.display import Javascript, display
    payload=base64.b64encode(Path(ARCHIVE).read_bytes()).decode()
    display(Javascript('const s=atob('+json.dumps(payload)+');const b=Uint8Array.from(s,c=>c.charCodeAt(0));const u=URL.createObjectURL(new Blob([b],{type:"application/zip"}));const a=document.createElement("a");a.href=u;a.download="giada_ie_p7_native_operational.zip";document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(u),10000);'))
